In [0]:
import requests
import json
from datetime import datetime

EVENTHUB_NAMESPACE = "PASTE_YOUR_NAMESPACE_STRING_HERE"
EVENTHUB_CONNECTION_STRING = "PASTE_YOUR_CONNECTION_STRING_HERE"
TOPIC = "wa-road-incidents"
FEED_NAME = "road_incidents"
API_URL = "https://services2.arcgis.com/cHGEnmsJ165IBJRM/arcgis/rest/services/WebEoc_RoadIncidents/FeatureServer/1/query"

# --- Step 1: read the watermark ---
watermark_row = spark.sql(
    f"SELECT last_update_date FROM incident_watermark WHERE feed_name = '{FEED_NAME}'"
).collect()[0]
watermark_date = watermark_row["last_update_date"]
print("Watermark:", watermark_date)

# --- Step 2: fetch from the API ---
params = {"where": "1=1", "outFields": "*", "f": "json", "outSR": 4326}
response = requests.get(API_URL, params=params)
features = response.json().get("features", [])
print(f"Total records from API: {len(features)}")

# --- Step 3: filter using the watermark ---
new_records = []
for feature in features:
    attrs = feature["attributes"]
    record_date = datetime.strptime(attrs["UpdateDate"], "%d/%m/%Y %H:%M:%S")
    if record_date > watermark_date:
        new_records.append(feature)

print(f"New records: {len(new_records)}")

# --- Step 4 & 5: push to Event Hubs and update watermark, only if there's something new ---
if new_records:
    json_records = [json.dumps(feature) for feature in new_records]
    df_to_send = spark.createDataFrame([(r,) for r in json_records], ["value"])

    kafka_options = {
        "kafka.bootstrap.servers": f"{EVENTHUB_NAMESPACE}:9093",
        "kafka.security.protocol": "SASL_SSL",
        "kafka.sasl.mechanism": "PLAIN",
        "kafka.sasl.jaas.config": f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="{EVENTHUB_CONNECTION_STRING}";',
        "topic": TOPIC
    }

    df_to_send.write.format("kafka").options(**kafka_options).save()
    print(f"Pushed {len(new_records)} records to Event Hubs")

    newest = max(new_records, key=lambda f: datetime.strptime(f["attributes"]["UpdateDate"], "%d/%m/%Y %H:%M:%S"))
    newest_date = datetime.strptime(newest["attributes"]["UpdateDate"], "%d/%m/%Y %H:%M:%S")
    newest_global_id = newest["attributes"]["GlobalID"]

    spark.sql(f"""
        UPDATE incident_watermark
        SET last_update_date = '{newest_date}', last_global_id = '{newest_global_id}', updated_at = current_timestamp()
        WHERE feed_name = '{FEED_NAME}'
    """)
    print("Watermark updated to:", newest_date)
else:
    print("No new records - watermark unchanged.")

Watermark: 2026-07-02 22:20:21
Total records from API: 18
New records: 0
No new records - watermark unchanged.
